In [65]:

import os
import re
import json
import requests
from bs4 import BeautifulSoup
from collections import OrderedDict

from debate_info import debate_info
from debate_dict import debate_dict
from info_urls import info_urls

ImportError: cannot import name 'info_urls' from 'debate_dict' (/Users/pulpdesign/Pulp Dropbox/Shah Ullah/My Mac (PULPs-MBP.lan)/Desktop/pulp-shah/usp-debates-2/inspection/debate_dict.py)

In [15]:
import pandas as pd

def get_raw():
    raw_info = []
    with open('../extraction/scraped_info.txt', 'r') as info_file:
        lines = info_file.readlines()
        num_lines = len(lines)
        i = 0
        year_debate_count = {}
        while i < num_lines:
            file_line = lines[i].strip()
            # Check if line contains a filename of the expected format
            if file_line.count('-') >= 2 and not file_line.endswith("-info.txt"): 
                # Extract year, month and day from filename
                year, month, day, _ = file_line.split('-')[:4]
                id = year_debate_count.get(year, 0) + 1
                year_debate_count[year] = id
                sentence_count = int(lines[i+1].split(': ')[1])
                word_count = int(lines[i+2].split(': ')[1])
                raw_info.append([year, month, day, id, word_count, sentence_count])
            i += 3  # Skip to the next file

    raw_df = pd.DataFrame(raw_info, columns=['YYYY', 'MM', 'DD', 'id', 'word_count', 'sentence_count'])
    return raw_df

raw_info = get_raw()


In [ ]:
raw_info.sort_values("YYYY")

In [ ]:
import os

def delete_raw_info_files(directory):
    for root, dirs, files in os.walk(directory):
        for file in files:
            if file == "raw-info.txt":
                file_path = os.path.join(root, file)
                os.remove(file_path)
                print(f"Deleted file: {file_path}")

# Example usage
directory_path = "../debates/precleaned_debates"
delete_raw_info_files(directory_path)

In [22]:
import os
import re
import pandas as pd

# Function to get the last name(s) from a full name
def get_last_name(full_name):
    parts = full_name.split()
    if len(parts) > 1 and parts[0].isupper() and parts[1].isupper():
        # Handle cases like 'VAN BUREN'
        return parts[0] + " " + parts[1]
    else:
        # Regular case, just return the last part of the name
        return parts[-1]

def get_speakers(file):
    with open(file, 'r') as f:
        lines = f.readlines()
    speakers = set()
    for line in lines:
        match = re.match(r"([A-Z\s]+?):", line)
        if match:
            speakers.add(match.group(1))
    return speakers

def get_last_names(directory):
    file_speaker_map = {}
    for subdir, dirs, files in os.walk(directory):
        for file in files:
            if file.endswith('.txt'):
                full_file_path = subdir + os.sep + file
                speakers = get_speakers(full_file_path)
                last_speakers = [get_last_name(speaker) for speaker in speakers]
                file_speaker_map[full_file_path] = last_speakers
    return file_speaker_map

In [ ]:
file_speaker_map = get_last_names('../debates/precleaned_debates') # replace with your directory

# Converting the dictionary into a pandas DataFrame
df = pd.DataFrame(list(file_speaker_map.items()), columns=['file', 'speakers'])
df['speaker_count'] = df['speakers'].apply(len)

df.sort_values("speaker_count")